# Loan Default Prediction
 
**Dataset:** 121,856 loan applications · 40 features · 8% default rate



#Setup & Imports


In [1]:
# ── Path setup
import sys, os

# This line adds the parent of 'notebooks/' (= project root) to Python path
# Adjust the number of '..' if your notebook is nested differently
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path includes src: {'src' in str(sys.path)}")

Project root: c:\Users\rahul\Documents\GitHub\loan_default_mlops
Python path includes src: True


In [4]:
# -------------------- Basic setup --------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------- Visualization config --------------------
plt.style.use("dark_background")

COLORS = {
    "primary": "#1A56DB",
    "success": "#16A34A",
    "danger": "#DC2626",
    "warning": "#D97706",
    "accent": "#06B6D4",
    "neutral": "#6B7280",
    "white": "#F8FAFC"
}

FIG_WIDE = (16, 5)
FIG_GRID = (16, 10)
FIG_TALL = (12, 8)

# -------------------- ML / Metrics --------------------
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.decomposition import PCA

# NOTE: make sure imbalanced-learn is installed in your environment
from imblearn.over_sampling import SMOTE

# -------------------- Project modules --------------------
# Keeping imports explicit so it's easy to trace dependencies
from data.ingest import (
    load_data,
    clean_and_cast,
    validate_data,
    split_data,
)

from features.feature_eng import (
    engineer_features,
    build_preprocessor,
    prepare_xy,
)

from models.train import (
    evaluate_model,
    get_feature_importance,
)

from monitoring.drift import (
    calculate_psi,
    ModelMonitor,
)

import joblib

# -------------------- Sanity check --------------------
if __name__ == "__main__":
    print("Environment ready")
    print(f"pandas  : {pd.__version__}")
    print(f"seaborn : {sns.__version__}")

ModuleNotFoundError: No module named 'imblearn'

In [ ]:
# ── Data paths ───────────────────────────────────────────────────────────────
# Adjust these paths if your working directory is different
RAW_PATH   = os.path.join(PROJECT_ROOT, 'data', 'raw',       'Dataset.csv')
TRAIN_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train.csv')
TEST_PATH  = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test.csv')
MODEL_PATH = os.path.join(PROJECT_ROOT, 'models', 'model.pkl')

print("Checking file paths...")
for name, path in [("Raw data", RAW_PATH), ("Train CSV", TRAIN_PATH),
                   ("Test CSV", TEST_PATH), ("Model pkl", MODEL_PATH)]:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING — run: python scripts/train_pipeline.py"
    print(f"  {name}: {status}")

---
<a id='2'></a>
## Section 2 — EDA: Data Quality & Distributions

**Where this sits:** Before any modelling. Run after confirming paths above.  
**What it does:** Loads raw data using your production `ingest.py`, then visualizes:
- Class imbalance
- Missing value patterns
- Distribution of key numeric features split by Default label
- Correlation heatmap

In [ ]:
# ── Load raw data using your production ingest module ────────────────────────
df_raw = load_data(RAW_PATH)
validate_data(df_raw)
df_clean = clean_and_cast(df_raw.copy())
df_clean = df_clean.drop(columns=['ID'], errors='ignore')

print(f"Shape: {df_clean.shape}")
print(f"Default rate: {df_clean['Default'].mean():.2%}")
df_clean.head(3)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 1: Class Imbalance — The Core Problem
# ════════════════════════════════════════════════════════════════════════════════
# WHY THIS CHART: First thing any interviewer asks — "how imbalanced is your data?"
# This chart proves you understood the problem before touching any model.

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_WIDE)
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 1 — Class Imbalance: The Core Challenge', 
             fontsize=14, fontweight='bold', color=PALETTE['white'], y=1.02)

counts    = df_clean['Default'].value_counts()
labels    = ['Non-Default (0)', 'Default (1)']
colors_pie = [PALETTE['teal'], PALETTE['red']]

# Pie chart
ax = axes[0]
ax.set_facecolor('#0F1E35')
wedges, texts, autotexts = ax.pie(
    counts.values, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90, explode=[0, 0.1],
    textprops={'color': PALETTE['white'], 'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold'); at.set_fontsize(12)
ax.set_title('Overall Distribution', color=PALETTE['white'], fontsize=11)

# Bar chart with count annotations
ax2 = axes[1]
ax2.set_facecolor('#0F1E35')
bars = ax2.bar(['Non-Default', 'Default'], counts.values, 
               color=colors_pie, edgecolor=PALETTE['white'], linewidth=0.8, width=0.5)
for bar, count in zip(bars, counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{count:,}', ha='center', va='bottom', 
             color=PALETTE['white'], fontsize=11, fontweight='bold')
ax2.set_title('Absolute Counts', color=PALETTE['white'], fontsize=11)
ax2.set_ylabel('Number of Applications', color=PALETTE['white'])
ax2.tick_params(colors=PALETTE['white'])
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

# What naive model achieves
ax3 = axes[2]
ax3.set_facecolor('#0F1E35')
naive_acc   = counts.iloc[0] / len(df_clean)
our_recall  = 0.70
strategies  = ['Naive Model\n(predict all safe)', 'Our GB Model\n(with SMOTE)']
accuracy_v  = [naive_acc, 0.88]
recall_v    = [0.00,       our_recall]
x = np.arange(len(strategies))
w = 0.3
ax3.bar(x - w/2, accuracy_v, w, label='Accuracy', color=PALETTE['blue'],   alpha=0.9)
ax3.bar(x + w/2, recall_v,   w, label='Recall',   color=PALETTE['amber'],  alpha=0.9)
for xi, (a, r) in zip(x, zip(accuracy_v, recall_v)):
    ax3.text(xi-w/2, a+0.01, f'{a:.0%}', ha='center', color=PALETTE['white'], fontsize=9, fontweight='bold')
    ax3.text(xi+w/2, r+0.01, f'{r:.0%}', ha='center', color=PALETTE['white'], fontsize=9, fontweight='bold')
ax3.set_ylim(0, 1.12)
ax3.set_xticks(x); ax3.set_xticklabels(strategies, color=PALETTE['white'], fontsize=9)
ax3.set_title('Why Accuracy is Misleading', color=PALETTE['white'], fontsize=11)
ax3.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=9)
ax3.tick_params(colors=PALETTE['white'])
ax3.spines['bottom'].set_color(PALETTE['gray'])
ax3.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_01_class_imbalance.png', dpi=150, bbox_inches='tight', 
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Key insight: 92% accuracy by predicting 'safe' for everyone = 0% recall = useless!")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 2: Missing Value Analysis
# ════════════════════════════════════════════════════════════════════════════════
# WHY THIS CHART: Shows data quality. Interviewers ask "how did you handle missing data?"
# This proves you investigated before choosing median imputation.

missing = (df_clean.isnull().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 2 — Missing Value Analysis', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

# Horizontal bar chart
ax = axes[0]
ax.set_facecolor('#0F1E35')
colors_bar = [PALETTE['red'] if v > 30 else PALETTE['amber'] if v > 10 
              else PALETTE['teal'] for v in missing.values]
bars = ax.barh(missing.index, missing.values, color=colors_bar, edgecolor='none')
ax.axvline(30, color=PALETTE['red'],   ls='--', lw=1.5, alpha=0.8, label='>30% = High missing')
ax.axvline(10, color=PALETTE['amber'], ls='--', lw=1.5, alpha=0.8, label='>10% = Moderate missing')
for bar, val in zip(bars, missing.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', color=PALETTE['white'], fontsize=8)
ax.set_xlabel('Missing %', color=PALETTE['white'])
ax.set_title('Missing Values by Column', color=PALETTE['white'], fontsize=11)
ax.tick_params(colors=PALETTE['white'], labelsize=8)
ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax.spines['bottom'].set_color(PALETTE['gray'])
ax.spines['left'].set_color(PALETTE['gray'])

# Missing by Default class
ax2 = axes[1]
ax2.set_facecolor('#0F1E35')
top5_missing = missing.head(5).index.tolist()
miss_by_class = df_clean.groupby('Default')[top5_missing].apply(
    lambda x: x.isnull().mean() * 100).T
miss_by_class.plot(kind='bar', ax=ax2, 
                   color=[PALETTE['teal'], PALETTE['red']],
                   edgecolor='none', width=0.6)
ax2.set_title('Missing Rate by Default Class\n(Top 5 features)', color=PALETTE['white'], fontsize=11)
ax2.set_xlabel('Feature', color=PALETTE['white'])
ax2.set_ylabel('Missing %', color=PALETTE['white'])
ax2.tick_params(colors=PALETTE['white'], labelsize=8, axis='x', rotation=30)
ax2.tick_params(colors=PALETTE['white'], axis='y')
ax2.legend(['Non-Default', 'Default'], facecolor=PALETTE['navy'], 
           labelcolor=PALETTE['white'], fontsize=9)
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_02_missing_values.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print(f"💡 {len(missing)} columns have missing values. Score_Source_3 is most missing ({missing.iloc[0]:.1f}%)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 3: Key Feature Distributions by Default Label
# ════════════════════════════════════════════════════════════════════════════════
# WHY THIS CHART: Shows which features separate defaulters from non-defaulters.
# This justifies which features are most important for the model.

key_features = ['Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Age_Days']
df_clean['age_years'] = df_clean['Age_Days'].abs() / 365

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 3 — Feature Distributions: Default vs Non-Default', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

for i, feat in enumerate(key_features):
    col = df_clean[feat].dropna()
    
    # Cap at 99th percentile to remove outliers from visualization
    cap = col.quantile(0.99)
    
    # Top row: KDE plot
    ax = axes[0][i]
    ax.set_facecolor('#0F1E35')
    for label, color, name in [(0, PALETTE['teal'], 'Non-Default'), 
                                (1, PALETTE['red'],  'Default')]:
        subset = df_clean[df_clean['Default'] == label][feat].dropna()
        subset = subset[subset <= cap]
        subset.plot.kde(ax=ax, color=color, lw=2.5, label=name)
        ax.axvline(subset.median(), color=color, ls='--', lw=1, alpha=0.7)
    ax.set_title(f'{feat}\n(KDE)', color=PALETTE['white'], fontsize=9, fontweight='bold')
    ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=7)
    ax.tick_params(colors=PALETTE['white'], labelsize=7)
    ax.set_ylabel('Density', color=PALETTE['white'], fontsize=8)
    ax.spines['bottom'].set_color(PALETTE['gray'])
    ax.spines['left'].set_color(PALETTE['gray'])
    
    # Bottom row: Box plot
    ax2 = axes[1][i]
    ax2.set_facecolor('#0F1E35')
    data_to_plot = [
        df_clean[df_clean['Default'] == 0][feat].dropna().clip(upper=cap).values,
        df_clean[df_clean['Default'] == 1][feat].dropna().clip(upper=cap).values,
    ]
    bp = ax2.boxplot(data_to_plot, patch_artist=True, widths=0.5,
                     medianprops={'color': PALETTE['white'], 'lw': 2})
    for patch, color in zip(bp['boxes'], [PALETTE['teal'], PALETTE['red']]):
        patch.set_facecolor(color + '66')
        patch.set_edgecolor(color)
    for element in ['whiskers', 'caps', 'fliers']:
        for item in bp[element]:
            item.set_color(PALETTE['gray'])
    ax2.set_xticklabels(['Non-Default', 'Default'], color=PALETTE['white'], fontsize=8)
    ax2.set_title(f'{feat}\n(Boxplot)', color=PALETTE['white'], fontsize=9)
    ax2.tick_params(colors=PALETTE['white'], labelsize=7)
    ax2.spines['bottom'].set_color(PALETTE['gray'])
    ax2.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_03_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Dashed vertical lines = medians. Wider separation between teal/red = better predictive power.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 4: Correlation Heatmap
# ════════════════════════════════════════════════════════════════════════════════
# WHY THIS CHART: Shows multicollinearity between features.
# Important for justifying feature selection and regularization choices.

numeric_cols = ['Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Age_Days',
                'Score_Source_1', 'Score_Source_2', 'Social_Circle_Default',
                'Credit_Bureau', 'Default']

df_corr = df_clean[numeric_cols].copy()
df_corr.columns = [c.replace('Client_', '').replace('_', '\n') for c in df_corr.columns]

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(PALETTE['navy'])
ax.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 4 — Feature Correlation Heatmap', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

cmap = sns.diverging_palette(220, 20, as_cmap=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(corr_matrix, ax=ax, mask=mask, cmap=cmap, center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8, 'color': PALETTE['white']},
            linewidths=0.5, linecolor='#1E293B',
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})

ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', 
                   color=PALETTE['white'], fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, 
                   color=PALETTE['white'], fontsize=8)

# Highlight Default column
for i, col in enumerate(corr_matrix.columns):
    if col == 'Default':
        ax.add_patch(plt.Rectangle((i, 0), 1, len(corr_matrix), 
                                   fill=False, edgecolor=PALETTE['amber'], lw=2))

plt.tight_layout()
plt.savefig('chart_04_correlation.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()

# Print top correlations with Default
corr_with_target = corr_matrix['Default'].abs().sort_values(ascending=False).iloc[1:]
print("Top features correlated with Default:")
print(corr_with_target.to_string())

---
<a id='3'></a>
## Section 3 — Feature Engineering Visualization

**Where this sits:** After EDA, before training.  
**What it does:** Shows the 4 engineered features and proves they have stronger signal than raw columns. Uses your production `feature_eng.py` directly.

In [ ]:
# ── Engineer features using your production module ───────────────────────────
df_featured = engineer_features(df_clean.copy())

engineered = ['income_annuity_ratio', 'credit_income_ratio', 'age_years', 'employment_years']
print("New columns created:", engineered)
print("\nSample values:")
df_featured[engineered + ['Default']].describe().round(2)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 5: Engineered Features — Separation Power
# ════════════════════════════════════════════════════════════════════════════════
# WHY THIS CHART: Proves that engineered features separate classes better than raw.
# income_annuity_ratio is a ratio of two raw columns — more informative than either alone.

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 5 — Engineered Features vs Raw Features: Separation Power',
             fontsize=13, fontweight='bold', color=PALETTE['white'])

pairs = [
    ('Client_Income',       'income_annuity_ratio', 'Raw Income',         'Income/Annuity Ratio'),
    ('Credit_Amount',       'credit_income_ratio',  'Raw Credit Amount',  'Credit/Income Ratio'),
    ('Age_Days',            'age_years',             'Raw Age (days)',      'Age (years)'),
    ('Employed_Days',       'employment_years',      'Raw Employment Days','Employment (years)'),
]

for col_i, (raw_col, eng_col, raw_label, eng_label) in enumerate(pairs):
    for row_i, (col_name, label) in enumerate([(raw_col, raw_label), (eng_col, eng_label)]):
        ax = axes[row_i][col_i]
        ax.set_facecolor('#0F1E35')
        
        for label_v, color, name in [(0, PALETTE['teal'], 'Non-Default'),
                                      (1, PALETTE['red'],  'Default')]:
            subset = df_featured[df_featured['Default'] == label_v][col_name].dropna()
            cap = subset.quantile(0.99)
            subset = subset[subset <= cap]
            subset.plot.kde(ax=ax, color=color, lw=2.5, label=name, fill=True, alpha=0.3)
        
        title_color = PALETTE['amber'] if row_i == 1 else PALETTE['white']
        prefix = '✨ ENGINEERED: ' if row_i == 1 else 'RAW: '
        ax.set_title(f'{prefix}{label}', color=title_color, fontsize=8, fontweight='bold')
        ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=6)
        ax.tick_params(colors=PALETTE['white'], labelsize=7)
        ax.set_ylabel('Density', color=PALETTE['white'], fontsize=7)
        ax.spines['bottom'].set_color(PALETTE['gray'])
        ax.spines['left'].set_color(PALETTE['gray'])

# Row labels
axes[0][0].set_ylabel('RAW FEATURES', color=PALETTE['white'], fontsize=9, fontweight='bold')
axes[1][0].set_ylabel('✨ ENGINEERED', color=PALETTE['amber'], fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_05_feature_engineering.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Better separation (less overlap between teal and red) = more predictive power")
print("   Engineered ratios (bottom row) typically show cleaner separation than raw values (top row)")

---
<a id='4'></a>
## Section 4 — SMOTE: Visualizing the Imbalance Fix

**Where this sits:** Before model training. Run after feature engineering.  
**What it does:** Visually demonstrates what SMOTE does in 2D (using PCA to reduce dimensions) and shows the before/after class distribution.

In [ ]:
# ── Load processed train data and apply SMOTE for visualization ──────────────
train_df = pd.read_csv(TRAIN_PATH)
X_train, y_train = prepare_xy(train_df)

# Build and fit preprocessor (same as production pipeline)
preprocessor = build_preprocessor()

# Filter to columns that exist
from features.feature_eng import NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES
X_train = engineer_features(X_train)

# Fit and transform
X_proc = preprocessor.fit_transform(X_train)

print(f"Training shape before SMOTE: {X_proc.shape}")
print(f"Class distribution: Non-Default={int((y_train==0).sum()):,}  Default={int((y_train==1).sum()):,}")

In [ ]:
# ── Apply SMOTE (same params as production) ──────────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_proc, y_train)

print(f"After SMOTE — Non-Default: {int((y_resampled==0).sum()):,}  Default: {int((y_resampled==1).sum()):,}")

# PCA to 2D for visualization (PCA reduces hundreds of features to 2 for plotting)
print("Reducing to 2D with PCA for visualization...")
pca = PCA(n_components=2, random_state=42)

# Sample for speed (PCA on full dataset takes time)
sample_size = min(5000, len(X_proc))
idx = np.random.RandomState(42).choice(len(X_proc), sample_size, replace=False)
X_2d_before = pca.fit_transform(X_proc[idx])
y_before     = y_train.values[idx]

# After SMOTE — sample from both real and synthetic
n_real   = len(X_proc)
n_synth  = len(X_resampled) - n_real
X_2d_after_real  = pca.transform(X_resampled[:n_real])[:2000]
X_2d_after_synth = pca.transform(X_resampled[n_real:])[:min(n_synth, 2000)]
y_after_real      = y_resampled[:n_real][:2000]

print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 6: SMOTE — Before and After in 2D PCA Space
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 6 — SMOTE: Before vs After in PCA Feature Space', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

# Panel 1: Before SMOTE
ax = axes[0]
ax.set_facecolor('#0F1E35')
mask0 = y_before == 0
mask1 = y_before == 1
ax.scatter(X_2d_before[mask0, 0], X_2d_before[mask0, 1], 
           c=PALETTE['teal'], s=8, alpha=0.4, label=f'Non-Default ({mask0.sum():,})')
ax.scatter(X_2d_before[mask1, 0], X_2d_before[mask1, 1], 
           c=PALETTE['red'],  s=15, alpha=0.8, label=f'Default ({mask1.sum():,})')
ax.set_title('BEFORE SMOTE\n(Imbalanced)', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8, markerscale=2)
ax.tick_params(colors=PALETTE['white'])
ax.set_xlabel('PCA Component 1', color=PALETTE['white'])
ax.set_ylabel('PCA Component 2', color=PALETTE['white'])
ax.spines['bottom'].set_color(PALETTE['gray'])
ax.spines['left'].set_color(PALETTE['gray'])

# Panel 2: After SMOTE — real + synthetic
ax2 = axes[1]
ax2.set_facecolor('#0F1E35')
mask_safe = y_after_real == 0
mask_real_default = y_after_real == 1
ax2.scatter(X_2d_after_real[mask_safe, 0],  X_2d_after_real[mask_safe, 1],
            c=PALETTE['teal'],   s=8, alpha=0.3, label='Non-Default (real)')
ax2.scatter(X_2d_after_real[mask_real_default, 0], X_2d_after_real[mask_real_default, 1],
            c=PALETTE['red'],    s=15, alpha=0.8, label='Default (real)')
ax2.scatter(X_2d_after_synth[:, 0], X_2d_after_synth[:, 1],
            c=PALETTE['amber'],  s=20, alpha=0.7, marker='*', label='Default (SYNTHETIC)')
ax2.set_title('AFTER SMOTE\n(Balanced — real + synthetic)', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax2.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=7, markerscale=2)
ax2.tick_params(colors=PALETTE['white'])
ax2.set_xlabel('PCA Component 1', color=PALETTE['white'])
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

# Panel 3: Class balance comparison
ax3 = axes[2]
ax3.set_facecolor('#0F1E35')
before = [int((y_train==0).sum()), int((y_train==1).sum())]
after  = [int((y_resampled==0).sum()), int((y_resampled==1).sum())]
x = np.arange(2)
bars1 = ax3.bar(x - 0.2, before, 0.35, label='Before SMOTE', 
                color=[PALETTE['teal'], PALETTE['red']], alpha=0.6, edgecolor=PALETTE['white'])
bars2 = ax3.bar(x + 0.2, after,  0.35, label='After SMOTE',  
                color=[PALETTE['teal'], PALETTE['amber']], alpha=0.9, edgecolor=PALETTE['white'])
for bars in [bars1, bars2]:
    for bar in bars:
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom',
                 color=PALETTE['white'], fontsize=8, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(['Non-Default', 'Default'], color=PALETTE['white'])
ax3.set_title('Class Balance\nBefore vs After SMOTE', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax3.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=9)
ax3.tick_params(colors=PALETTE['white'])
ax3.set_ylabel('Count', color=PALETTE['white'])
ax3.spines['bottom'].set_color(PALETTE['gray'])
ax3.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_06_smote_visualization.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Gold stars (★) = synthetic SMOTE samples. They fill in the sparse red cluster,")
print("   forcing the model to learn the default region of feature space thoroughly.")

---
<a id='5'></a>
## Section 5 — Model Training & Evaluation

**Where this sits:** After `python scripts/train_pipeline.py` has been run.  
**What it does:** Loads the saved `model.pkl`, evaluates on the test set, and produces all standard ML evaluation charts. **This is the section you open during the interview.**

In [ ]:
# ── Load saved model and test data ──────────────────────────────────────────
model_pipeline = joblib.load(MODEL_PATH)
test_df = pd.read_csv(TEST_PATH)
test_df = engineer_features(test_df)

X_test, y_test = prepare_xy(test_df)

# Get predictions
y_proba = model_pipeline.predict_proba(X_test)[:, 1]
y_pred  = (y_proba > 0.3).astype(int)   # our tuned threshold

# Core metrics
auc    = roc_auc_score(y_test, y_proba)
recall = recall_score(y_test, y_pred)
prec   = precision_score(y_test, y_pred)
f1     = f1_score(y_test, y_pred)
cm     = confusion_matrix(y_test, y_pred)

print("=" * 45)
print("  MODEL PERFORMANCE SUMMARY")
print("=" * 45)
print(f"  AUC-ROC   : {auc:.4f}   (target > 0.70)")
print(f"  Recall    : {recall:.4f}   (target > 0.60) ← PRIMARY")
print(f"  Precision : {prec:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("=" * 45)
print(f"  Confusion Matrix:")
print(f"    TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"    FN={cm[1,0]:,}  TP={cm[1,1]:,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 7: ROC Curve + Precision-Recall Curve
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 7 — ROC Curve & Precision-Recall Curve', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

# ROC Curve
ax = axes[0]
ax.set_facecolor('#0F1E35')
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color=PALETTE['blue'], lw=2.5, label=f'Our Model (AUC = {auc:.3f})')
ax.plot([0,1],[0,1], color=PALETTE['gray'], ls='--', lw=1.5, label='Random (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.15, color=PALETTE['blue'])
# Mark our operating point at threshold=0.3
threshold = 0.3
pred_03 = (y_proba > threshold).astype(int)
fpr_pt  = (pred_03[y_test==0] == 1).mean()
tpr_pt  = recall_score(y_test, pred_03)
ax.scatter([fpr_pt], [tpr_pt], s=150, c=PALETTE['amber'], zorder=5, 
           edgecolors=PALETTE['white'], label=f'Threshold=0.3 (Recall={tpr_pt:.2f})')
ax.set_xlabel('False Positive Rate', color=PALETTE['white'])
ax.set_ylabel('True Positive Rate', color=PALETTE['white'])
ax.set_title('ROC Curve', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax.tick_params(colors=PALETTE['white'])
ax.spines['bottom'].set_color(PALETTE['gray'])
ax.spines['left'].set_color(PALETTE['gray'])
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

# Precision-Recall Curve
ax2 = axes[1]
ax2.set_facecolor('#0F1E35')
precision_c, recall_c, thresholds_c = precision_recall_curve(y_test, y_proba)
pr_auc = np.trapz(precision_c[::-1], recall_c[::-1])
ax2.plot(recall_c, precision_c, color=PALETTE['teal'], lw=2.5, 
         label=f'Our Model (PR-AUC ≈ {pr_auc:.3f})')
ax2.axhline(y_test.mean(), color=PALETTE['gray'], ls='--', lw=1.5,
            label=f'Baseline (default rate = {y_test.mean():.2f})')
ax2.fill_between(recall_c, precision_c, alpha=0.15, color=PALETTE['teal'])
# Operating point
prec_pt = precision_score(y_test, pred_03)
ax2.scatter([tpr_pt], [prec_pt], s=150, c=PALETTE['amber'], zorder=5,
            edgecolors=PALETTE['white'], label=f'Threshold=0.3 (P={prec_pt:.2f}, R={tpr_pt:.2f})')
ax2.set_xlabel('Recall', color=PALETTE['white'])
ax2.set_ylabel('Precision', color=PALETTE['white'])
ax2.set_title('Precision-Recall Curve\n(Better for imbalanced datasets)', 
              color=PALETTE['white'], fontsize=11, fontweight='bold')
ax2.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax2.tick_params(colors=PALETTE['white'])
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])
ax2.set_xlim([0,1]); ax2.set_ylim([0,1.02])

plt.tight_layout()
plt.savefig('chart_07_roc_pr_curves.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Gold dot = our operating point at threshold=0.3.")
print("   PR-AUC is more informative than ROC-AUC for imbalanced datasets.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 8: Confusion Matrix Heatmap (Annotated)
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 8 — Confusion Matrix: What Types of Errors We Make',
             fontsize=14, fontweight='bold', color=PALETTE['white'])

for ax_i, (thresh, label) in enumerate([(0.5, 'Default Threshold = 0.5'), 
                                          (0.3, 'Our Threshold = 0.3')]):
    ax = axes[ax_i]
    ax.set_facecolor('#0F1E35')
    y_p = (y_proba > thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_p)
    
    cell_labels = np.array([
        [f'TN\n{cm_t[0,0]:,}\nCorrect Safe', f'FP\n{cm_t[0,1]:,}\nFalse Alarm'],
        [f'FN\n{cm_t[1,0]:,}\nMISSED\nDEFAULT!',  f'TP\n{cm_t[1,1]:,}\nCaught\nDefault']
    ])
    cell_colors = np.array([
        [PALETTE['teal']+'88',  PALETTE['amber']+'88'],
        [PALETTE['red']+'cc',   PALETTE['green']+'88'],
    ])
    
    for row in range(2):
        for col in range(2):
            ax.add_patch(plt.Rectangle((col, row), 1, 1, 
                                       facecolor=cell_colors[1-row][col], 
                                       edgecolor=PALETTE['white'], lw=2))
            ax.text(col+0.5, 1-row+0.5, cell_labels[1-row][col], 
                    ha='center', va='center', color=PALETTE['white'],
                    fontsize=9, fontweight='bold', linespacing=1.5)
    
    recall_t = recall_score(y_test, y_p)
    prec_t   = precision_score(y_test, y_p)
    ax.set_title(f'{label}\nRecall={recall_t:.2f}  Precision={prec_t:.2f}', 
                 color=PALETTE['amber'] if thresh==0.3 else PALETTE['white'],
                 fontsize=10, fontweight='bold')
    ax.set_xlim([0,2]); ax.set_ylim([0,2])
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(['Pred: Safe', 'Pred: Risky'], 
                                                     color=PALETTE['white'], fontsize=9)
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(['Actual: Risky', 'Actual: Safe'],
                                                     color=PALETTE['white'], fontsize=9, rotation=90,
                                                     va='center')
    ax.tick_params(length=0)
    ax.spines['bottom'].set_color(PALETTE['gray'])
    ax.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_08_confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()
print("💡 Compare left vs right: threshold=0.3 has more FP (false alarms) but MUCH fewer FN (missed defaults).")
print("   FN is the expensive error — bank loses the full loan amount for each one.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 9: Feature Importance
# ════════════════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(PALETTE['navy'])
ax.set_facecolor('#0F1E35')
fig.suptitle('Chart 9 — Feature Importance (Gradient Boosting)', 
             fontsize=14, fontweight='bold', color=PALETTE['white'])

# Extract feature names from ColumnTransformer
try:
    clf = model_pipeline.named_steps['classifier']
    importances = clf.feature_importances_
    
    # Get feature names: numeric first, then cat, then binary (same order as ColumnTransformer)
    from features.feature_eng import NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES
    all_feature_names = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
    n_features = len(importances)
    feature_names = all_feature_names[:n_features]
    
    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True).tail(20)
    
    colors_bar = [PALETTE['amber'] if 'ratio' in f or 'years' in f else 
                  PALETTE['blue'] for f in feat_imp.index]
    
    bars = ax.barh(feat_imp.index, feat_imp.values, color=colors_bar, edgecolor='none', height=0.7)
    
    for bar, val in zip(bars, feat_imp.values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', color=PALETTE['white'], fontsize=8)
    
    # Legend
    legend_patches = [
        mpatches.Patch(color=PALETTE['amber'], label='Engineered features (our additions)'),
        mpatches.Patch(color=PALETTE['blue'],  label='Original raw features'),
    ]
    ax.legend(handles=legend_patches, facecolor=PALETTE['navy'], 
              labelcolor=PALETTE['white'], fontsize=9, loc='lower right')
    
    ax.set_title('Top 20 Most Important Features', color=PALETTE['white'], fontsize=11)
    ax.set_xlabel('Importance Score', color=PALETTE['white'])
    ax.tick_params(colors=PALETTE['white'], labelsize=8)
    ax.spines['bottom'].set_color(PALETTE['gray'])
    ax.spines['left'].set_color(PALETTE['gray'])
    
    plt.tight_layout()
    plt.savefig('chart_09_feature_importance.png', dpi=150, bbox_inches='tight',
                facecolor=PALETTE['navy'])
    plt.show()
    
    engineered_in_top10 = [f for f in feat_imp.tail(10).index if 'ratio' in f or 'years' in f]
    print(f"💡 Engineered features in top 10: {engineered_in_top10}")
    print("   If engineered features appear near the top, our feature engineering added real value.")

except Exception as e:
    print(f"Note: Could not extract importances — model may have SMOTE step. Error: {e}")
    print("Tip: Try model_pipeline.named_steps to inspect available step names.")

---
<a id='6'></a>
## Section 6 — Threshold Tuning

**Where this sits:** After evaluation, before deployment decision.  
**What it does:** Shows the complete trade-off between Recall, Precision, and F1 as threshold changes. This is the chart you use to justify choosing 0.3 over the default 0.5.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 10: Threshold vs Recall / Precision / F1 + Business Cost
# ════════════════════════════════════════════════════════════════════════════════

thresholds = np.arange(0.05, 0.75, 0.01)
recalls, precisions, f1s = [], [], []

for t in thresholds:
    y_p = (y_proba > t).astype(int)
    recalls.append(recall_score(y_test, y_p, zero_division=0))
    precisions.append(precision_score(y_test, y_p, zero_division=0))
    f1s.append(f1_score(y_test, y_p, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 10 — Threshold Tuning: Finding the Optimal Business Decision Point',
             fontsize=13, fontweight='bold', color=PALETTE['white'])

# Left: metric curves
ax = axes[0]
ax.set_facecolor('#0F1E35')
ax.plot(thresholds, recalls,    color=PALETTE['red'],   lw=2.5, label='Recall')
ax.plot(thresholds, precisions, color=PALETTE['teal'],  lw=2.5, label='Precision')
ax.plot(thresholds, f1s,        color=PALETTE['amber'], lw=2.5, label='F1 Score')

# Mark our choice
ax.axvline(0.3, color=PALETTE['white'], ls='--', lw=2, alpha=0.8)
ax.text(0.3, 0.05, 'OUR CHOICE
0.3', ha='center', color=PALETTE['white'], 
        fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=PALETTE['blue']+'88', 
                  edgecolor=PALETTE['white']))

# Shade zones
ax.axvspan(0.05, 0.20, alpha=0.1, color=PALETTE['red'],  label='Too aggressive')
ax.axvspan(0.50, 0.75, alpha=0.1, color=PALETTE['teal'], label='Too conservative')

# Mark where curves intersect
idx_intersect = np.argmin(np.abs(np.array(recalls) - np.array(precisions)))
ax.axvline(thresholds[idx_intersect], color=PALETTE['purple'], ls=':', lw=1.5,
           label=f'P=R at t={thresholds[idx_intersect]:.2f}')

ax.set_xlabel('Classification Threshold', color=PALETTE['white'], fontsize=10)
ax.set_ylabel('Score', color=PALETTE['white'], fontsize=10)
ax.set_title('Recall / Precision / F1 Trade-off', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax.tick_params(colors=PALETTE['white'])
ax.set_xlim([0.05, 0.75]); ax.set_ylim([0, 1.05])
ax.grid(color='#1E293B', lw=0.5)
ax.spines['bottom'].set_color(PALETTE['gray'])
ax.spines['left'].set_color(PALETTE['gray'])

# Right: Business cost curve
ax2 = axes[1]
ax2.set_facecolor('#0F1E35')

# Cost model: missing a default costs ₹5L, false alarm costs ₹0 (they reapply)
COST_FN  = 500000   # cost per missed defaulter (₹5 lakh)
COST_FP  = 0        # cost per falsely declined (they go elsewhere, minimal direct loss)
N_APPS   = 10000    # monthly applications

total_defaults     = int(y_test.sum() / len(y_test) * N_APPS)
total_safe         = N_APPS - total_defaults

total_costs = []
prevented_losses = []
for t in thresholds:
    y_p   = (y_proba > t).astype(int)
    fn    = int((1 - recall_score(y_test, y_p, zero_division=0)) * total_defaults)
    fp    = int((1 - precision_score(y_test, y_p, zero_division=1)) * 
                int((y_proba > t).mean() * N_APPS))
    cost  = fn * COST_FN + fp * COST_FP
    prevented = (total_defaults - fn) * COST_FN
    total_costs.append(cost / 1e7)        # in crores
    prevented_losses.append(prevented / 1e7)

ax2.plot(thresholds, total_costs,     color=PALETTE['red'],   lw=2.5, label='Residual loss (₹ Crore)')
ax2.plot(thresholds, prevented_losses, color=PALETTE['green'], lw=2.5, label='Prevented loss (₹ Crore)')
ax2.axvline(0.3, color=PALETTE['white'], ls='--', lw=2, alpha=0.8, label='Our threshold (0.3)')
ax2.fill_between(thresholds, total_costs, alpha=0.15, color=PALETTE['red'])
ax2.fill_between(thresholds, prevented_losses, alpha=0.15, color=PALETTE['green'])

ax2.set_xlabel('Classification Threshold', color=PALETTE['white'], fontsize=10)
ax2.set_ylabel('₹ Crore (per 10K applications)', color=PALETTE['white'], fontsize=10)
ax2.set_title('Business Cost Impact\n(₹5L per missed default)', color=PALETTE['white'], fontsize=11, fontweight='bold')
ax2.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax2.tick_params(colors=PALETTE['white'])
ax2.set_xlim([0.05, 0.75])
ax2.grid(color='#1E293B', lw=0.5)
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

plt.tight_layout()
plt.savefig('chart_10_threshold_tuning.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()

# Print decision table
print("\n📊 Threshold Decision Table (per 10,000 applications):")
print(f"{'Threshold':>10} | {'Recall':>7} | {'Precision':>9} | {'Defaults Caught':>15} | {'Residual Loss':>13}")
print("-" * 65)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    y_p  = (y_proba > t).astype(int)
    rec  = recall_score(y_test, y_p, zero_division=0)
    prec = precision_score(y_test, y_p, zero_division=0)
    caught = int(rec * total_defaults)
    loss   = int((1 - rec) * total_defaults) * COST_FN
    marker = " ← OUR CHOICE" if t == 0.3 else ""
    print(f"{t:>10.1f} | {rec:>7.2f} | {prec:>9.2f} | {caught:>15,} | ₹{loss/1e7:>8.1f}Cr{marker}")

---
<a id='7'></a>
## Section 7 — Monitoring: PSI Drift Detection

**Where this sits:** After deployment. Run periodically in production.  
**What it does:** Uses your production `monitoring/drift.py` module to visualize PSI scores across all features and demonstrate what drift looks like visually.

In [ ]:
# ── Set up monitoring using production module ────────────────────────────────
train_df_mon = pd.read_csv(TRAIN_PATH)
test_df_mon  = pd.read_csv(TEST_PATH)

# Use test as "stable production" and a modified version as "drifted production"
prod_stable  = test_df_mon.copy()
prod_drifted = test_df_mon.copy()

# Simulate drift: shift income and credit amount distributions
np.random.seed(42)
prod_drifted['Client_Income']  = pd.to_numeric(prod_drifted['Client_Income'],  errors='coerce') * 2.5
prod_drifted['Credit_Amount']  = pd.to_numeric(prod_drifted['Credit_Amount'],  errors='coerce') * 1.8
prod_drifted['Loan_Annuity']   = pd.to_numeric(prod_drifted['Loan_Annuity'],   errors='coerce') * 2.0

# Calculate PSI for each numeric column
numeric_cols_mon = ['Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Age_Days',
                    'Score_Source_1', 'Score_Source_2']

def get_psi_scores(ref_df, prod_df, cols):
    scores = {}
    for col in cols:
        ref  = pd.to_numeric(ref_df[col],  errors='coerce').dropna().values
        prod = pd.to_numeric(prod_df[col], errors='coerce').dropna().values
        if len(ref) > 10 and len(prod) > 10:
            scores[col] = calculate_psi(ref, prod)
    return scores

psi_stable  = get_psi_scores(train_df_mon, prod_stable,  numeric_cols_mon)
psi_drifted = get_psi_scores(train_df_mon, prod_drifted, numeric_cols_mon)

print("PSI Scores — Stable Production vs Drifted Production:")
print(f"{'Feature':30s} | {'Stable PSI':10s} | {'Drifted PSI':11s} | {'Status'}")
print("-" * 70)
for col in numeric_cols_mon:
    s = psi_stable.get(col, 0)
    d = psi_drifted.get(col, 0)
    status = "🔴 CRITICAL" if d > 0.25 else "🟡 WARNING" if d > 0.10 else "🟢 STABLE"
    print(f"{col:30s} | {s:10.4f} | {d:11.4f} | {status}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 11: PSI Dashboard — Stable vs Drifted
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor(PALETTE['navy'])
fig.suptitle('Chart 11 — PSI Monitoring Dashboard: Stable vs Drifted Production',
             fontsize=13, fontweight='bold', color=PALETTE['white'])

# Panel 1: PSI bar chart comparison
ax = axes[0][0]
ax.set_facecolor('#0F1E35')
features  = list(psi_stable.keys())
stable_v  = [psi_stable[f]  for f in features]
drifted_v = [psi_drifted[f] for f in features]
x = np.arange(len(features))
w = 0.35
ax.bar(x - w/2, stable_v,  w, label='Stable (PSI)',  color=PALETTE['teal'], alpha=0.9, edgecolor=PALETTE['white'], lw=0.5)
ax.bar(x + w/2, drifted_v, w, label='Drifted (PSI)', color=PALETTE['red'],  alpha=0.9, edgecolor=PALETTE['white'], lw=0.5)
ax.axhline(0.10, color=PALETTE['amber'], ls='--', lw=1.5, label='Warning (0.10)')
ax.axhline(0.25, color=PALETTE['red'],   ls='--', lw=2.0, label='Critical (0.25)')
ax.set_xticks(x)
ax.set_xticklabels([f.replace('Client_', '').replace('_', '
') for f in features],
                   color=PALETTE['white'], fontsize=8)
ax.set_ylabel('PSI Score', color=PALETTE['white'])
ax.set_title('PSI per Feature: Stable vs Drifted', color=PALETTE['white'], fontsize=10, fontweight='bold')
ax.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=7)
ax.tick_params(colors=PALETTE['white'])
ax.spines['bottom'].set_color(PALETTE['gray'])
ax.spines['left'].set_color(PALETTE['gray'])

# Panel 2: Income distribution comparison (the most drifted feature)
ax2 = axes[0][1]
ax2.set_facecolor('#0F1E35')
ref_income  = pd.to_numeric(train_df_mon['Client_Income'],   errors='coerce').dropna()
stab_income = pd.to_numeric(prod_stable['Client_Income'],    errors='coerce').dropna()
drft_income = pd.to_numeric(prod_drifted['Client_Income'],   errors='coerce').dropna()

cap = ref_income.quantile(0.99)
ref_income.clip(upper=cap).plot.kde( ax=ax2, color=PALETTE['blue'],  lw=2.5, label=f'Training (ref) PSI=0')
stab_income.clip(upper=cap).plot.kde(ax=ax2, color=PALETTE['teal'],  lw=2.0, ls='--', label=f"Stable  PSI={psi_stable['Client_Income']:.2f}")
drft_income.clip(upper=cap).plot.kde(ax=ax2, color=PALETTE['red'],   lw=2.0, ls='--', label=f"Drifted PSI={psi_drifted['Client_Income']:.2f}")
ax2.set_title('Client Income Distribution\n(Training vs Production)', color=PALETTE['white'], fontsize=10, fontweight='bold')
ax2.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
ax2.set_xlabel('Client Income', color=PALETTE['white'])
ax2.tick_params(colors=PALETTE['white'])
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

# Panel 3: PSI gauge — status indicator
ax3 = axes[1][0]
ax3.set_facecolor('#0F1E35')
ax3.set_xlim(0,10); ax3.set_ylim(0,10); ax3.axis('off')
ax3.set_title('Current System Status', color=PALETTE['white'], fontsize=10, fontweight='bold')

statuses = [
    (1.0, 7.5, 'STABLE SCENARIO', PALETTE['green'],  'ALL features PSI < 0.10
No action required'),
    (1.0, 4.0, 'DRIFTED SCENARIO', PALETTE['red'],   f"Client_Income PSI={psi_drifted['Client_Income']:.2f}
Credit_Amount PSI={psi_drifted['Credit_Amount']:.2f}
→ RETRAIN TRIGGERED"),
]
for (x,y,title,col,desc) in statuses:
    ax3.add_patch(FancyBboxPatch((x,y-1.0), 8.0, 2.0, boxstyle='round,pad=0.1',
                                  facecolor=col+'22', edgecolor=col, lw=2.5))
    ax3.text(5.0, y+0.5, title, ha='center', color=col, fontsize=10, fontweight='bold')
    ax3.text(5.0, y-0.4, desc,  ha='center', va='center', color=PALETTE['white'], fontsize=8.5, linespacing=1.4)

# Panel 4: Prediction score drift
ax4 = axes[1][1]
ax4.set_facecolor('#0F1E35')
# Get prediction probabilities on stable vs "drifted" inputs
try:
    X_stable  = engineer_features(pd.read_csv(TEST_PATH))
    X_stable_feat, _ = prepare_xy(X_stable)
    proba_stable = model_pipeline.predict_proba(X_stable_feat)[:, 1]
    
    # Simulate drifted predictions by perturbing input slightly
    X_drift_feat = X_stable_feat.copy()
    if 'Client_Income' in X_drift_feat.columns:
        X_drift_feat['Client_Income'] = pd.to_numeric(X_drift_feat['Client_Income'], errors='coerce') * 2.5
    proba_drifted = model_pipeline.predict_proba(X_drift_feat)[:, 1]
    
    psi_pred = calculate_psi(proba_stable, proba_drifted)
    
    pd.Series(proba_stable).plot.kde( ax=ax4, color=PALETTE['blue'], lw=2.5, label=f'Stable predictions')
    pd.Series(proba_drifted).plot.kde(ax=ax4, color=PALETTE['red'],  lw=2.5, ls='--', label=f'Drifted predictions')
    ax4.axvline(0.3, color=PALETTE['amber'], ls='--', lw=1.5, alpha=0.8, label='Decision threshold')
    ax4.set_title(f'Prediction Score Distribution\nPSI = {psi_pred:.3f}', 
                  color=PALETTE['white'], fontsize=10, fontweight='bold')
    ax4.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=8)
    ax4.set_xlabel('Predicted Default Probability', color=PALETTE['white'])
    ax4.tick_params(colors=PALETTE['white'])
    ax4.spines['bottom'].set_color(PALETTE['gray'])
    ax4.spines['left'].set_color(PALETTE['gray'])
except Exception as e:
    ax4.text(0.5, 0.5, f'Run train_pipeline.py first
{str(e)[:60]}', 
             ha='center', va='center', transform=ax4.transAxes, color=PALETTE['amber'])

plt.tight_layout()
plt.savefig('chart_11_psi_monitoring.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()

---
<a id='8'></a>
## Section 8 — Business Summary Dashboard

**Where this sits:** Final section — this is what you show to business stakeholders.  
**What it does:** Translates model outputs into business decisions and financial impact. No ML jargon — pure business language.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHART 12: Business Decision Dashboard
# ════════════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor(PALETTE['navy'])
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Chart 12 — Business Dashboard: Model Impact on Loan Portfolio',
             fontsize=14, fontweight='bold', color=PALETTE['white'], y=1.01)

# ── Risk tier distribution ──────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#0F1E35')
low_risk    = (y_proba < 0.30).mean()
med_risk    = ((y_proba >= 0.30) & (y_proba < 0.60)).mean()
high_risk   = (y_proba >= 0.60).mean()
tier_vals   = [low_risk, med_risk, high_risk]
tier_labels = [f'AUTO APPROVE
{low_risk:.1%}', f'HUMAN REVIEW
{med_risk:.1%}', f'AUTO DECLINE
{high_risk:.1%}']
tier_colors = [PALETTE['teal'], PALETTE['amber'], PALETTE['red']]
wedges, texts = ax1.pie(tier_vals, colors=tier_colors, startangle=90, 
                         wedgeprops={'edgecolor': PALETTE['navy'], 'linewidth': 2})
ax1.legend(tier_labels, loc='lower center', facecolor=PALETTE['navy'],
           labelcolor=PALETTE['white'], fontsize=7.5, framealpha=0.8)
ax1.set_title('Risk Tier Distribution
(All Applications)', 
              color=PALETTE['white'], fontsize=10, fontweight='bold')

# ── Monthly financial impact ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#0F1E35')
N_MONTHLY      = 10000
DEFAULT_RATE   = 0.08
AVG_LOAN       = 500000   # ₹5 lakh
monthly_def    = int(N_MONTHLY * DEFAULT_RATE)
caught_def     = int(monthly_def * recall)
missed_def     = monthly_def - caught_def
saved          = caught_def * AVG_LOAN / 1e7     # crores
lost           = missed_def * AVG_LOAN / 1e7
categories     = ['Defaults
Caught', 'Defaults
Missed', 'Loss
Prevented (₹Cr)', 'Residual
Loss (₹Cr)']
values         = [caught_def, missed_def, saved, lost]
colors_b       = [PALETTE['green'], PALETTE['red'], PALETTE['teal'], PALETTE['amber']]
bars = ax2.bar(categories, values, color=colors_b, edgecolor=PALETTE['white'], lw=0.8, width=0.5)
for bar, val, cat in zip(bars, values, categories):
    prefix = '₹' if 'Cr' in cat else ''
    suffix = 'Cr' if 'Cr' in cat else ''
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{prefix}{val:.0f}{suffix}', ha='center', va='bottom',
             color=PALETTE['white'], fontsize=9, fontweight='bold')
ax2.set_title('Monthly Impact
(10,000 applications)', 
              color=PALETTE['white'], fontsize=10, fontweight='bold')
ax2.tick_params(colors=PALETTE['white'], labelsize=8)
ax2.set_ylabel('Count / ₹ Crore', color=PALETTE['white'])
ax2.spines['bottom'].set_color(PALETTE['gray'])
ax2.spines['left'].set_color(PALETTE['gray'])

# ── Model performance summary card ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor('#0F1E35')
ax3.set_xlim(0,10); ax3.set_ylim(0,10); ax3.axis('off')
ax3.set_title('Model KPI Summary', color=PALETTE['white'], fontsize=10, fontweight='bold')

kpis = [
    ('AUC-ROC',   f'{auc:.3f}',    PALETTE['blue'],   '(>0.70 target)'),
    ('Recall',    f'{recall:.3f}',  PALETTE['red'],    '← PRIMARY KPI'),
    ('Precision', f'{prec:.3f}',    PALETTE['teal'],   ''),
    ('F1 Score',  f'{f1:.3f}',      PALETTE['amber'],  ''),
    ('Threshold', '0.300',          PALETTE['purple'], '(tuned)'),
]
for i, (name, val, col, note) in enumerate(kpis):
    y_pos = 9.0 - i * 1.7
    ax3.add_patch(FancyBboxPatch((0.3, y_pos-0.6), 9.4, 1.2,
                                  boxstyle='round,pad=0.08', facecolor=col+'22', edgecolor=col, lw=1.5))
    ax3.text(1.0, y_pos, name, va='center', color=PALETTE['white'], fontsize=10, fontweight='bold')
    ax3.text(6.5, y_pos, val, va='center', color=col, fontsize=14, fontweight='bold')
    ax3.text(8.2, y_pos, note, va='center', color=PALETTE['gray'], fontsize=7.5)

# ── Probability score histogram ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
ax4.set_facecolor('#0F1E35')
bins = np.linspace(0, 1, 31)
ax4.hist(y_proba[y_test==0], bins=bins, alpha=0.6, color=PALETTE['teal'], 
         label='Non-Default (actual)', density=True)
ax4.hist(y_proba[y_test==1], bins=bins, alpha=0.7, color=PALETTE['red'],  
         label='Default (actual)',     density=True)
ax4.axvline(0.3, color=PALETTE['amber'], ls='--', lw=2.5, label='Decision threshold = 0.3')
ax4.axvline(0.5, color=PALETTE['gray'],  ls=':',  lw=1.5, label='Default threshold = 0.5')
ax4.fill_betweenx([0, ax4.get_ylim()[1] if ax4.get_ylim()[1] > 0 else 10], 
                  0, 0.3, alpha=0.08, color=PALETTE['teal'])
ax4.fill_betweenx([0, ax4.get_ylim()[1] if ax4.get_ylim()[1] > 0 else 10], 
                  0.3, 0.6, alpha=0.08, color=PALETTE['amber'])
ax4.fill_betweenx([0, ax4.get_ylim()[1] if ax4.get_ylim()[1] > 0 else 10], 
                  0.6, 1.0, alpha=0.08, color=PALETTE['red'])
ax4.text(0.15, 0.8, 'AUTO
APPROVE', transform=ax4.transAxes, 
         color=PALETTE['teal'], fontsize=9, ha='center', alpha=0.7)
ax4.set_xlabel('Predicted Default Probability', color=PALETTE['white'], fontsize=10)
ax4.set_ylabel('Density', color=PALETTE['white'], fontsize=10)
ax4.set_title('Score Distribution by Actual Label
(well-separated = good model)', 
              color=PALETTE['white'], fontsize=10, fontweight='bold')
ax4.legend(facecolor=PALETTE['navy'], labelcolor=PALETTE['white'], fontsize=9)
ax4.tick_params(colors=PALETTE['white'])
ax4.spines['bottom'].set_color(PALETTE['gray'])
ax4.spines['left'].set_color(PALETTE['gray'])

# ── Annual savings ─────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor('#0F1E35')
ax5.set_xlim(0,10); ax5.set_ylim(0,10); ax5.axis('off')
ax5.set_title('Annual Business Value', color=PALETTE['white'], fontsize=10, fontweight='bold')

annual_saved  = saved  * 12
annual_lost   = lost   * 12
annual_net    = annual_saved - annual_lost

metrics_fin = [
    ('Annual Prevented Loss',  f'₹{annual_saved:.0f} Cr',  PALETTE['green']),
    ('Annual Residual Loss',   f'₹{annual_lost:.0f} Cr',   PALETTE['red']),
    ('Net Benefit',            f'₹{annual_net:.0f} Cr/yr', PALETTE['amber']),
    ('Auto-Processed',         f'{low_risk:.0%} of apps',   PALETTE['teal']),
    ('Human Review Queue',     f'{med_risk:.0%} of apps',   PALETTE['purple']),
]
for i, (label, val, col) in enumerate(metrics_fin):
    y_pos = 9.2 - i * 1.7
    ax5.add_patch(FancyBboxPatch((0.2, y_pos-0.65), 9.6, 1.2,
                                  boxstyle='round,pad=0.08', facecolor=col+'22', edgecolor=col, lw=1.5))
    ax5.text(0.7, y_pos, label, va='center', color=PALETTE['white'], fontsize=8)
    ax5.text(7.0, y_pos, val, va='center', color=col, fontsize=11, fontweight='bold')

plt.savefig('chart_12_business_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['navy'])
plt.show()

print(f"\n💼 BUSINESS SUMMARY (per 10,000 monthly applications):")
print(f"   Defaults caught    : {caught_def:,} / {monthly_def:,}")
print(f"   Loss prevented     : ₹{saved:.1f} Crore/month")
print(f"   Residual loss      : ₹{lost:.1f} Crore/month")
print(f"   Auto-approved      : {low_risk:.0%} (instant, no staff)")
print(f"   Human review queue : {med_risk:.0%} (analysts focus here only)")

---
## 📍 Where to Place This Notebook in Your Project

```
loan_default_mlops/
├── notebooks/                          ← CREATE THIS FOLDER
│   └── loan_default_eda_visualization.ipynb  ← PUT THIS FILE HERE
│
├── src/
│   ├── data/ingest.py          ← Notebook imports from here (Section 1-2)
│   ├── features/feature_eng.py ← Notebook imports from here (Section 3-4)
│   ├── models/train.py         ← Notebook imports from here (Section 5-6)
│   └── monitoring/drift.py     ← Notebook imports from here (Section 7)
│
├── data/
│   ├── raw/Dataset.csv         ← Raw data (Section 2)
│   └── processed/
│       ├── train.csv           ← Generated by train_pipeline.py
│       └── test.csv            ← Generated by train_pipeline.py
│
├── models/
│   └── model.pkl               ← Generated by train_pipeline.py
│
└── scripts/
    └── train_pipeline.py       ← RUN THIS FIRST before opening notebook

```

## 🚀 How to Run

```bash
# Step 1: Run the training pipeline first (generates data and model)
python scripts/train_pipeline.py

# Step 2: Launch Jupyter
cd loan_default_mlops
jupyter notebook notebooks/loan_default_eda_visualization.ipynb

# Or use JupyterLab
jupyter lab
```

## ⚠️ Important Rules

1. **Never copy code from src/ into this notebook** — always import it
2. **Never put plt.show() or visualization code in src/ files** — only in notebooks
3. **Run cells in order** — each section depends on the one above
4. **Run train_pipeline.py first** — Sections 5-8 need model.pkl and processed CSVs

## 🎤 Interview Usage

**Open this notebook when the interviewer asks:**
- "Show me your EDA" → Section 2
- "How did you handle imbalance?" → Section 4 (SMOTE PCA chart)
- "What model did you choose and why?" → Section 5 (ROC curve + feature importance)
- "How did you pick the threshold?" → Section 6 (threshold tuning chart)
- "How do you monitor in production?" → Section 7 (PSI dashboard)
- "What's the business value?" → Section 8 (business dashboard)